In [3]:
"""
Actions that we need to train the adapter on:
- LGTM - Accept the draft as-is
- add content
- add tool call at a specific index
- replace content
- replace tool call 
- delete tool call at a specific index
- delete content

How should we modify the data so that the adapter can perform the above actions?
- LGTM: keep data as-is
- add content: delete content when gold content is present.
- add tool call: drop tool call at random indices
- replace content: corrupt the content.
- replace tool call:
    - corrupt the tool call name at index I.
    - corrupt the tool call arguments at index I.
- delete tool call: add random tool call at random indices.
- delete content: add content to when no gold content is present.

Sometimes the adapter model can take multiple actions in a single turn. Some actions cannot be performed together.
- no other action allowed if LGTM is performed.
- you can either add content or replace content or delete content but not do them together.
- you cannot add a tool call at index I and delete a tool call at the same index I. Should do replace tool call at index I.
"""

'\nActions that we need to train the adapter on:\n- LGTM - Accept the draft as-is\n- add content\n- add tool call at a specific index\n- replace content\n- replace tool call \n- delete tool call at a specific index\n- delete content\n\nHow should we modify the data so that the adapter can perform the above actions?\n- LGTM: keep data as-is\n- add content: delete content when gold content is present.\n- add tool call: drop tool call at random indices\n- replace content: corrupt the content.\n- replace tool call:\n    - corrupt the tool call name at index I.\n    - corrupt the tool call arguments at index I.\n- delete tool call: add random tool call at random indices.\n- delete content: add content to when no gold content is present.\n\nSometimes the adapter model can take multiple actions in a single turn. Some actions cannot be performed together.\n- no other action allowed if LGTM is performed.\n- you can either add content or replace content or delete content but not do them together

In [108]:
# building training data for the adapter

import json

hermes_data_path = './data/raw/hermes_func_calling.jsonl'
hermes_data = []
with open(hermes_data_path, 'r') as f:
    for line in f:
        hermes_data.append(json.loads(line))

hermes_data[0]



{'source_dataset': 'NousResearch/hermes-function-calling-v1',
 'source_config': 'func_calling',
 'source_row_idx': 0,
 'id': '82088305-310b-45cb-ac76-ab273503b5cd',
 'conversations': [{'from': 'system',
   'value': 'You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.\n<tools>\n[{"type": "function", "function": {"name": "get_camera_live_feed", "description": "Retrieves the live feed from a specified security camera.", "parameters": {"type": "object", "properties": {"camera_id": {"type": "string", "description": "The unique identifier for the camera."}, "stream_quality": {"type": "string", "description": "The desired quality of the live stream.", "enum": ["720p", "1080p", "4k"]}}, "required": ["camera_id"]}}}, {"type": "function", "function": {"name": "list_all_cameras", "description": "Lists all t

In [109]:
# convert to messages format
from typing import Any

ROLE_MAP = {
    "system": "system",
    "human": "user",
    "gpt": "assistant",
    "tool": "tool",
}

def _coerce_message(turn: dict[str, Any]) -> dict[str, str] | None:
    from_role = turn.get("from")
    value = turn.get("value")
    if not isinstance(from_role, str) or not isinstance(value, str):
        return None

    role = ROLE_MAP.get(from_role)
    if role is None:
        return None
    return {"role": role, "content": value}

_coerce_message(hermes_data[0]['conversations'][0])

{'role': 'system',
 'content': 'You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.\n<tools>\n[{"type": "function", "function": {"name": "get_camera_live_feed", "description": "Retrieves the live feed from a specified security camera.", "parameters": {"type": "object", "properties": {"camera_id": {"type": "string", "description": "The unique identifier for the camera."}, "stream_quality": {"type": "string", "description": "The desired quality of the live stream.", "enum": ["720p", "1080p", "4k"]}}, "required": ["camera_id"]}}}, {"type": "function", "function": {"name": "list_all_cameras", "description": "Lists all the security cameras connected to the home network.", "parameters": {"type": "object", "properties": {"include_offline": {"type": "boolean", "description": "Whether to include cameras t

In [110]:
for example in hermes_data:
    messages = []
    for turn in example['conversations']:
        message = _coerce_message(turn)
        if message is not None:
            messages.append(message)
    example['messages'] = messages
hermes_data[0]

{'source_dataset': 'NousResearch/hermes-function-calling-v1',
 'source_config': 'func_calling',
 'source_row_idx': 0,
 'id': '82088305-310b-45cb-ac76-ab273503b5cd',
 'conversations': [{'from': 'system',
   'value': 'You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don\'t make assumptions about what values to plug into functions.\n<tools>\n[{"type": "function", "function": {"name": "get_camera_live_feed", "description": "Retrieves the live feed from a specified security camera.", "parameters": {"type": "object", "properties": {"camera_id": {"type": "string", "description": "The unique identifier for the camera."}, "stream_quality": {"type": "string", "description": "The desired quality of the live stream.", "enum": ["720p", "1080p", "4k"]}}, "required": ["camera_id"]}}}, {"type": "function", "function": {"name": "list_all_cameras", "description": "Lists all t

In [111]:
# check if all system prompt start with "You are a function calling AI model"
for example in hermes_data:
    if example['messages'][0]['role'] == 'system' and not example['messages'][0]['content'].startswith('You are a function calling AI model'):
        print(example['messages'][0]['content'])

You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema. 
<tools>
[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of open-ended questions related to the document, that are potentially ambiguous.", "parameters": {"type": "object", "properties": {"open_ended_questions": {"type": "array", "items": {"type": "string"}}}, "required": ["open_ended_questions"]}}}]
</tools>
For each extraction function call return a json object with function name and arguments followed by a <tool_call> tag with the following schema:
<tool_call>
{"name": <function-name>, "arguments": <args-dict>}
</tool_call>
You are an expert structured information extraction AI model. You will be provided with do

In [112]:
# check if all system prompt end with 
# <tool_call>
# {"name": <function-name>, "arguments": <args-dict>}
# </tool_call>

s = """
<tool_call>
{"name": <function-name>, "arguments": <args-dict>}
</tool_call>
""".strip()

for example in hermes_data:
    if example['messages'][0]['role'] == 'system' and not example['messages'][0]['content'].strip().endswith(s):
        print(example['messages'][0]['content'])

In [113]:
# Unique ways to telling the model to output a tool call
unique_tool_call_prompts = set()
for example in hermes_data:
    if example['messages'][0]['role'] == 'system':
        prompt = example['messages'][0]['content'].strip().splitlines()[-4]
        unique_tool_call_prompts.add(prompt)

len(unique_tool_call_prompts)

2

In [114]:
# unique system prompts
unique_system_prompts = set()
for example in hermes_data:
    if example['messages'][0]['role'] == 'system':
        unique_system_prompts.add(example['messages'][0]['content'].strip())

len(unique_system_prompts)

1113

In [115]:
len(hermes_data)

1893

In [116]:
for x in unique_system_prompts:
    print(x)
    break



You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions.
<tools>
[{"type": "function", "function": {"name": "manage_artist_lineup", "description": "Manages the lineup of artists for an event, including stage assignments and performance times.", "parameters": {"type": "object", "properties": {"festival_id": {"type": "string", "description": "The unique identifier of the festival."}, "artists": {"type": "array", "description": "A list of artists with their performance details.", "items": {"type": "object", "properties": {"artist_id": {"type": "string", "description": "The unique identifier of the artist."}, "stage": {"type": "string", "description": "The stage where the artist will perform."}, "performance_time": {"type": "string", "description": "The scheduled start time of the artist's performance in I

In [117]:
print(next(iter(unique_system_prompts)))

You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions.
<tools>
[{"type": "function", "function": {"name": "manage_artist_lineup", "description": "Manages the lineup of artists for an event, including stage assignments and performance times.", "parameters": {"type": "object", "properties": {"festival_id": {"type": "string", "description": "The unique identifier of the festival."}, "artists": {"type": "array", "description": "A list of artists with their performance details.", "items": {"type": "object", "properties": {"artist_id": {"type": "string", "description": "The unique identifier of the artist."}, "stage": {"type": "string", "description": "The stage where the artist will perform."}, "performance_time": {"type": "string", "description": "The scheduled start time of the artist's performance in I

In [118]:
# drop tools
# drop tool call prompt

import re
ptrn = re.compile(r"<tools>\n.*\n<\/tools>", flags=re.DOTALL)

# This is a formatting error in the system prompt for hermes. we will drop these examples 
bad_s = """You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema. {"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that ask how ideas in the document 
are connected or relate to each other. These identify relationships between concepts.", "parameters":"""


cleaned_examples = []
for example in hermes_data:
    system_prompt = example['messages'][0]['content']
    original_system_prompt = system_prompt

    # drop tool call prompt
    system_prompt = system_prompt.strip().splitlines()[:-4]
    system_prompt = "\n".join(system_prompt)

    # drop tool calls
    just_before_dropping_tool_calls = system_prompt
    system_prompt = ptrn.sub("", system_prompt).strip()
    if system_prompt.strip() == bad_s: continue

    example['messages'][0]['content'] = system_prompt
    cleaned_examples.append(example)
    
len(hermes_data), len(cleaned_examples)

(1893, 1832)

In [119]:
example['messages'][0]['content']

"You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema."

In [73]:
ptrn

re.compile(r'<tools>\n.*\n<\/tools>', re.DOTALL|re.UNICODE)

In [120]:
# convert tool def into openai compatible tool def

tool_call_ptrn = re.compile(r"<tools>\n(.*)\n<\/tools>", flags=re.DOTALL)
for example in cleaned_examples:
    original_system_prompt = example['conversations'][0]['value']

    matches = re.search(tool_call_ptrn, original_system_prompt)
    if matches is None or len(matches.groups()) < 1: raise ValueError('matches is None')
    group = matches.groups()[0]
    group = group.strip()

    if not group: raise ValueError('group is empty')
    if '\n' in group: raise ValueError('group contains newlines. didn\'t expect that.')

    # tools are openai compatible tool defs
    tools = json.loads(group)
    example['tools'] = tools

In [121]:
tools

[{'type': 'function',
  'function': {'name': 'ExpertQAExtractor',
   'description': 'Extracts a list of questions that focus on summarizing a specific topic found in the document.',
   'parameters': {'type': 'object',
    'properties': {'topic_summarization_questions': {'type': 'array',
      'items': {'type': 'string'}}},
    'required': ['topic_summarization_questions']}}}]

In [86]:
example['messages'][1]['content']

'I\'ve recently installed a new security system at my home, and I want to ensure everything is functioning as it should. Specifically, I\'d like to start by checking the live feed from the camera located at the front door to monitor any activity. The camera has a unique identifier, which I\'ve already configured to be "front_door." I\'d prefer to view the live stream in high definition, so a 1080p quality would be ideal. Could you please call the appropriate function to retrieve the live feed from my front door camera in 1080p quality and provide me with the link to the stream?\n\nFollowing this, I would also like to record the live feed from this camera for the next 30 minutes. This is to test the recording feature and to keep an archived copy for security purposes. Please initiate the recording function for the "front_door" camera with a recording duration of 30 minutes.\n\nLastly, as part of my routine surveillance checks, I need to review footage from yesterday between 3 PM and 5 P

In [91]:
from dotenv import load_dotenv; load_dotenv()
from openai import OpenAI
import os

client = OpenAI(base_url='https://api.together.xyz/v1', api_key=os.environ['TOGETHER_API_KEY'])
response = client.chat.completions.create(
    model='zai-org/GLM-4.5-Air-FP8',
    messages=[
        {"role": "user", "content": example['messages'][1]['content']}
    ],
    tools=tools,
    tool_choice='auto',
)
print(response.choices[0].message)

ChatCompletionMessage(content="I'll help you manage your home security camera feeds by executing all three requested functions. Let me process each request step by step.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_869b3bc5525b48eea862b4f8', function=Function(arguments='{"camera_id": "front_door", "stream_quality": "1080p"}', name='get_camera_live_feed'), type='function', index=-1), ChatCompletionMessageFunctionToolCall(id='call_14f01b80a2bc40b3b52c7e15', function=Function(arguments='{"camera_id": "front_door", "duration": 30}', name='record_camera_feed'), type='function', index=-1), ChatCompletionMessageFunctionToolCall(id='call_6a140c4f06e74afc82ba197e', function=Function(arguments='{"camera_id": "front_garden", "start_time": "2023-04-22T15:00:00", "end_time": "2023-04-22T17:00:00"}', name='get_recorded_feed'), type='function', index=-1)], reasoning='The user is asking me to perform thre

In [93]:
response.choices[0].message.tool_calls

[ChatCompletionMessageFunctionToolCall(id='call_869b3bc5525b48eea862b4f8', function=Function(arguments='{"camera_id": "front_door", "stream_quality": "1080p"}', name='get_camera_live_feed'), type='function', index=-1),
 ChatCompletionMessageFunctionToolCall(id='call_14f01b80a2bc40b3b52c7e15', function=Function(arguments='{"camera_id": "front_door", "duration": 30}', name='record_camera_feed'), type='function', index=-1),
 ChatCompletionMessageFunctionToolCall(id='call_6a140c4f06e74afc82ba197e', function=Function(arguments='{"camera_id": "front_garden", "start_time": "2023-04-22T15:00:00", "end_time": "2023-04-22T17:00:00"}', name='get_recorded_feed'), type='function', index=-1)]

In [94]:
example = hermes_data[0]
for x in example['messages']:
    if x['role'] == 'assistant':
        print(x['content'])
        break





<tool_call>
{"name": "get_camera_live_feed", "arguments": {"camera_id": "front_door", "stream_quality": "1080p"}}
</tool_call>
<tool_call>
{"name": "record_camera_feed", "arguments": {"camera_id": "front_door", "duration": 30}}
</tool_call>
<tool_call>
{"name": "get_recorded_feed", "arguments": {"camera_id": "front_garden", "start_time": "2023-04-22T15:00:00Z", "end_time": "2023-04-22T17:00:00Z"}}
</tool_call>



In [127]:
from copy import deepcopy

separated_examples = []
for example in cleaned_examples:
    # count 'assistant' messages
    for i, x in enumerate(example['messages']):
        if x['role'] == 'assistant':
            _tmp = deepcopy(example)
            _tmp['messages'] = _tmp['messages'][:i+1]
            separated_examples.append(_tmp)

len(cleaned_examples), len(separated_examples)


(1832, 2759)

# Corrupt the messages

In [128]:
# convert to hf dataset obj
import pandas as pd

df = pd.DataFrame(separated_examples)
df.head()


,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages
0,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func..."
1,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func..."
2,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func..."
3,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func..."
4,NousResearch/hermes-function-calling-v1,func_calling,2,b099e2c1-8459-4848-bd4f-0385e0d45d59,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'se...",IoT and Home Automation,Thermostat Control,Adjust Smart Thermostat Settings,"[{'role': 'system', 'content': 'You are a func..."


In [129]:
# duplicate rows N times 
print(df.shape)
df = pd.concat([df] * 10, ignore_index=True)
print(df.shape)
df.head()


(2759, 10)
(27590, 10)


,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages
0,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func..."
1,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func..."
2,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func..."
3,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func..."
4,NousResearch/hermes-function-calling-v1,func_calling,2,b099e2c1-8459-4848-bd4f-0385e0d45d59,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'se...",IoT and Home Automation,Thermostat Control,Adjust Smart Thermostat Settings,"[{'role': 'system', 'content': 'You are a func..."


In [130]:
# add a new column to the dataset that will be used to determine the type of corruption

import numpy as np

AVAILABLE_CORRUPTION_TYPES = ['no_corruption', 'delete_content', 'corrupt_content', 'add_unnecessary_content', 'delete_tool_call', 'corrupt_tool_call_name', 'corrupt_tool_call_argument', 'add_extra_tool_call']
# for now give equal probability to each corruption type
df['corruption_type'] = np.random.choice(AVAILABLE_CORRUPTION_TYPES, size=df.shape[0])
df.head()

,source_dataset,source_config,source_row_idx,id,conversations,tools,category,subcategory,task,messages,corruption_type
0,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func...",delete_content
1,NousResearch/hermes-function-calling-v1,func_calling,0,82088305-310b-45cb-ac76-ab273503b5cd,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'ge...",IoT and Home Automation,Security Camera Management,View and Manage Security Camera Feeds,"[{'role': 'system', 'content': 'You are a func...",delete_tool_call
2,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func...",corrupt_tool_call_name
3,NousResearch/hermes-function-calling-v1,func_calling,1,bd48c5c1-186a-4120-be4a-61974d22fc8b,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'in...",IoT and Home Automation,Smart Home Setup,Set Up a Smart Home System,"[{'role': 'system', 'content': 'You are a func...",delete_tool_call
4,NousResearch/hermes-function-calling-v1,func_calling,2,b099e2c1-8459-4848-bd4f-0385e0d45d59,"[{'from': 'system', 'value': 'You are a functi...","[{'type': 'function', 'function': {'name': 'se...",IoT and Home Automation,Thermostat Control,Adjust Smart Thermostat Settings,"[{'role': 'system', 'content': 'You are a func...",corrupt_content


In [148]:
# use llms for corrupting the messages
# corruption types for llms: corrupt_content, add_unnecessary_content, corrupt_tool_call_name, corrupt_tool_call_argument, add_extra_tool_call

CORRUPTION_PROMPT_BASE = """
You are generating a deliberately corrupted assistant draft for adapter-training.
You will receive:
- the original system prompt with tool definitions if any
- the original conversation between the user and the assistant
- the gold assistant response
""".strip()

CORRUPTION_SYSTEM_PROMPTS = {
    "corrupt_content": "Corrupt the content of the assistant response.",
    "add_unnecessary_content": "Produce a draft with extra assistant text that should not be there.",
    "corrupt_tool_call_name": "Rewrite the given tool call with a wrong tool name.",
    "corrupt_tool_call_argument": "Rewrite the given tool call with wrong argument names or values or value types.",
    "add_extra_tool_call": "Produce an extra unnecessary tool call.",
}


CORRUPTION_PROMPT_RESPONSE_FORMAT = """
<corruption_output>
    <notes>[...]</notes>
    <content>if corrupted content</content>
    <tool_calls>if corrupted tool call</tool_calls>
</corruption_output>
"""


In [136]:
example

{'source_dataset': 'NousResearch/hermes-function-calling-v1',
 'source_config': 'func_calling',
 'source_row_idx': 1892,
 'id': '98c8fda0-ca02-4d3c-ac96-c5bd6bf6904a',
 'conversations': [{'from': 'system',
   'value': 'You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don\'t make assumptions about what values to plug into json schema. \n<tools>\n[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that focus on summarizing a specific topic found in the document.", "parameters": {"type": "object", "properties": {"topic_summarization_questions": {"type": "array", "items": {"type": "string"}}}, "required": ["topic_summarization_questions"]}}}]\n</tools>\nFor each extraction function call return a json object with function nam

In [135]:
messages = example['messages']
system_prompt = example['messages'][0]['content']
system_prompt += f'\n\n<tools>\n{json.dumps(example["tools"])}\n</tools>'
print(system_prompt)

You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema.

<tools>
[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that focus on summarizing a specific topic found in the document.", "parameters": {"type": "object", "properties": {"topic_summarization_questions": {"type": "array", "items": {"type": "string"}}}, "required": ["topic_summarization_questions"]}}}]
</tools>


In [139]:
CONVERSATION_HISTORY = []
for msg in messages[1:-1]:  # ignore system prompt and final assistant response
    if msg['role'] == 'user' or msg['role'] == 'tool':
        CONVERSATION_HISTORY.append(f'{msg["role"]}: {msg["content"]}')
    if msg['role'] == 'assistant':
        s = f'{msg["role"]}: {msg["content"]}'
        if 'tool_calls' in msg:
            s += f'\n<tool_calls>\n{json.dumps(msg["tool_calls"])}\n</tool_calls>'
        CONVERSATION_HISTORY.append(s)

print('\n---\n'.join(CONVERSATION_HISTORY))

user: Can you help me extract queries from the following passage <passage> = - 3 x + y - z = 0 
C50+^ A three-digit number has two properties. The tens-digit and the ones-digit add up to 5. If the number is written with the digits in the reverse order, and then subtracted 
SSS S L E B e e z e r : A F i r s t C o u r s e i n L i n e a r A l g e b r a 16 
from the original number, the result is 792. Use a system of equations to find all of the three-digit numbers with these properties. 
C51+^ Find all of the six-digit numbers in which the first digit is one less than the second, the third digit is half the second, the fourth digit is three times the third and the last two digits form a number that equals the sum of the fourth and fifth. The sum of all the digits is 24. (From The MENSA Puzzle Calendar for January 9, 2006.) 
C52+^ Driving along, Terry notices that the last four digits on his car's odometer are palindromic. A mile later, the last five digits are palindromic. After driving a

In [143]:
assistant_content_to_corrupt = messages[-1].get('content', None)
assistant_tool_calls_to_corrupt = messages[-1].get('tool_calls', None)

print(assistant_content_to_corrupt)
print(assistant_tool_calls_to_corrupt)


<tool_call>\n{"arguments": {"queries": ['How is a matrix defined and what is its purpose in linear algebra?', 'What is the difference between a coefficient matrix and a vector of constants in a system of equations?', 'How is a system of equations represented in matrix notation?'], "name": "ExpertQAExtractor"}}\n</tool_call>
None


In [149]:
prompt = f"""system_prompt:
{system_prompt}

conversation:
{CONVERSATION_HISTORY}

gold_assistant_response:
{assistant_content_to_corrupt}
"""

corruption_prompt = f"""
{CORRUPTION_PROMPT_BASE}

{CORRUPTION_SYSTEM_PROMPTS['corrupt_tool_call_name']}
Response format:
{CORRUPTION_PROMPT_RESPONSE_FORMAT}
"""
print(corruption_prompt)
print('\n---\n')
print(prompt)


You are generating a deliberately corrupted assistant draft for adapter-training.
You will receive:
- the original system prompt with tool definitions if any
- the original conversation between the user and the assistant
- the gold assistant response

Rewrite the given tool call with a wrong tool name.
Response format:

<corruption_output>
    <notes>[...]</notes>
    <content>if corrupted content</content>
    <tool_calls>if corrupted tool call</tool_calls>
</corruption_output>



---

system_prompt:
You are an expert structured information extraction AI model. You will be provided with documents to extract information from. You are also provided with the json schema to output extracted information in the function signatures within XML tags <tools></tools>. Don't make assumptions about what values to plug into json schema.

<tools>
[{"type": "function", "function": {"name": "ExpertQAExtractor", "description": "Extracts a list of questions that focus on summarizing a specific topic fo

In [ ]:
response = client.chat.completions.create(
    model='zai-org/GLM-4.7',
    messages=[
        {"role": "system", "content": corruption_prompt},
        {"role": "user", "content": prompt}
    ],
    max_tokens=32672,
)
print('corruption response:')
print(response.choices[0].message.content)

In [152]:
response.choices[0].finish_reason

'length'

In [ ]:
from __future__ import annotations

from typing import Literal, Any
from pydantic import BaseModel, ConfigDict, Field


CorruptionType = Literal[
    "delete_content",
    "corrupt_content",
    "add_unnecessary_content",
    "delete_tool_call",
    "corrupt_tool_call_name",
    "corrupt_tool_call_argument",
    "add_extra_tool_call",
]


class CorruptedToolFunction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str
    arguments: str  # must stay a JSON-serialized string


class CorruptedToolCall(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str
    type: Literal["function"]
    function: CorruptedToolFunction


class CorruptionOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    corruption_type: CorruptionType
    content: str
    tool_calls: list[CorruptedToolCall] = Field(default_factory=list)
    changed_tool_call_indices: list[int] = Field(default_factory=list)
    notes: str


CORRUPTION_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "corruption_output",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "corruption_type": {
                    "type": "string",
                    "enum": [
                        "delete_content",
                        "corrupt_content",
                        "add_unnecessary_content",
                        "delete_tool_call",
                        "corrupt_tool_call_name",
                        "corrupt_tool_call_argument",
                        "add_extra_tool_call",
                    ],
                },
                "content": {"type": "string"},
                "tool_calls": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "id": {"type": "string"},
                            "type": {"type": "string", "enum": ["function"]},
                            "function": {
                                "type": "object",
                                "additionalProperties": False,
                                "properties": {
                                    "name": {"type": "string"},
                                    "arguments": {"type": "string"},
                                },
                                "required": ["name", "arguments"],
                            },
                        },
                        "required": ["id", "type", "function"],
                    },
                },
                "changed_tool_call_indices": {
                    "type": "array",
                    "items": {"type": "integer", "minimum": 0},
                },
                "notes": {"type": "string"},
            },
            "required": [
                "corruption_type",
                "content",
                "tool_calls",
                "changed_tool_call_indices",
                "notes",
            ],
        },
    },
}


CORRUPTION_PROMPT_BASE = """
You are generating a deliberately corrupted assistant draft for adapter-training.

You will receive:
- the original conversation
- the gold assistant response as `content` and `tool_calls`
- the available tool definitions if any
- a target corruption type

Your job is to produce exactly one corrupted draft matching that corruption type.

Global rules:
- Preserve the overall structure of the gold response unless the target corruption requires changing it.
- Apply only the requested corruption type.
- Do not introduce any extra corruption types.
- Keep the response realistic and plausible.
- Return valid JSON only matching the required schema.
- `tool_calls` must stay in OpenAI chat-completions format.
- `function.arguments` must remain a JSON-serialized string.
- If a tool call is not being changed, preserve it exactly.
- If content is not being changed, preserve it exactly.
""".strip()


CORRUPTION_SYSTEM_PROMPTS = {
    "delete_content": CORRUPTION_PROMPT_BASE + """

Target corruption: delete_content

Produce a draft where correct assistant text content has been removed.
Rules:
- Set `content` to an empty string.
- Keep `tool_calls` identical to gold.
- Use this only when gold content is non-empty.
- `changed_tool_call_indices` must be empty.
- `notes` should briefly say that content was deleted.
""",
    "corrupt_content": CORRUPTION_PROMPT_BASE + """

Target corruption: corrupt_content

Produce a draft where the assistant text content is still present but wrong.
Rules:
- Keep `content` non-empty.
- Change only the text content.
- The corrupted text should be plausible but materially different from gold.
- Keep `tool_calls` identical to gold.
- `changed_tool_call_indices` must be empty.
- `notes` should briefly describe how the text was corrupted.
""",
    "add_unnecessary_content": CORRUPTION_PROMPT_BASE + """

Target corruption: add_unnecessary_content

Produce a draft with extra assistant text that should not be there.
Rules:
- Add non-empty `content`.
- Keep `tool_calls` identical to gold.
- Best used when gold content is empty or tool-only.
- The added text should be plausible but unnecessary.
- `changed_tool_call_indices` must be empty.
- `notes` should briefly say unnecessary content was added.
""",
    "delete_tool_call": CORRUPTION_PROMPT_BASE + """

Target corruption: delete_tool_call

Produce a draft where one required tool call has been removed.
Rules:
- Remove exactly one existing tool call.
- Keep remaining tool calls in the same relative order.
- Keep `content` identical to gold.
- Use only when gold has at least one tool call.
- `changed_tool_call_indices` should identify the missing position in the gold ordering.
- `notes` should briefly say which tool call was deleted.
""",
    "corrupt_tool_call_name": CORRUPTION_PROMPT_BASE + """

Target corruption: corrupt_tool_call_name

Produce a draft where exactly one tool call has the wrong function name.
Rules:
- Keep the number of tool calls unchanged.
- Change exactly one `function.name`.
- Do not change that tool call's arguments unless unavoidable.
- Keep `content` identical to gold.
- `changed_tool_call_indices` must contain the modified index.
- `notes` should briefly say the tool name was corrupted.
""",
    "corrupt_tool_call_argument": CORRUPTION_PROMPT_BASE + """

Target corruption: corrupt_tool_call_argument

Produce a draft where exactly one tool call has wrong arguments.
Rules:
- Keep the number of tool calls unchanged.
- Change exactly one `function.arguments` string.
- Keep the arguments syntactically valid JSON serialized as a string.
- Do not change the tool name unless unavoidable.
- Keep `content` identical to gold.
- `changed_tool_call_indices` must contain the modified index.
- `notes` should briefly say the arguments were corrupted.
""",
    "add_extra_tool_call": CORRUPTION_PROMPT_BASE + """

Target corruption: add_extra_tool_call

Produce a draft where one extra tool call has been inserted.
Rules:
- Add exactly one extra tool call.
- The extra tool call should be plausible but not part of the gold answer.
- Preserve all original tool calls exactly.
- Keep `content` identical to gold.
- `changed_tool_call_indices` must contain the inserted index.
- `notes` should briefly say an extra tool call was added.
""",
}